# Norm Provenance Ledger – полная разметка (поправка v0.4.4): 71 трасса, два судья

Оставшиеся 71 трасса подтверждающего набора (39 τ² + 32 MAST); 30 трасс расширенной проверки уже размечены и используются как есть. Оба судьи – GPT-5.6 Terra и GPT-5.6 Sol, бесплатные в Docent. Ячейки 1–5 ничего не отправляют.

## 1. Установка SDK (версия закреплена)

In [ ]:
!pip -q install docent==0.1.87
import docent, importlib.metadata as im
print('docent', im.version('docent'))

## 2. Загрузка архива
После запуска нажми «Выбрать файлы» и выбери `norm_ledger_v044.zip`.

In [ ]:
from google.colab import files
import zipfile, os, pathlib
up = files.upload()
name = next(n for n in up if n.endswith('.zip'))
zipfile.ZipFile(name).extractall('/content')
ROOT = pathlib.Path('/content/norm_ledger_v044')
os.chdir(ROOT)
print('рабочая папка:', ROOT)
print(sorted(p.name for p in ROOT.iterdir()))

## 3. Офлайн-проверка (без сети)
Должно вывести 71 прогон и 4203 блока.

In [ ]:
!python protocol/docent_run.py dry-run --corpus full71

## 4. Ключ API
Способ 1 (надёжнее): иконка ключа слева → Secrets → добавить `DOCENT_API_KEY`, включить доступ для блокнота.
Способ 2: ячейка сама спросит ключ скрытым полем. Ключ не печатается и не сохраняется в блокноте.

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ['DOCENT_API_KEY'] = userdata.get('DOCENT_API_KEY')
    print('ключ взят из Secrets')
except Exception:
    from getpass import getpass
    os.environ['DOCENT_API_KEY'] = getpass('DOCENT_API_KEY: ')
    print('ключ введён вручную')
print('ключ задан:', bool(os.environ.get('DOCENT_API_KEY')))

## 5. Какие модели доступны
Первый список – судьи, доступные без собственного ключа провайдера, как они записаны в SDK. Второй – то, что сервер отдаёт для «readings» (может не совпадать со списком судей). Выбери одну модель и впиши её в ячейку 6 в виде `provider:model:effort`.

In [ ]:
from docent.sdk import Docent
from docent._llm_util.providers.preference_types import PUBLIC_PROVIDER_PREFERENCES
print('Судьи по умолчанию в SDK 0.1.87:')
for m in PUBLIC_PROVIDER_PREFERENCES.default_judge_models:
    print(f'  {m.provider}:{m.model_name}:{m.reasoning_effort}')
try:
    c = Docent(api_key=os.environ['DOCENT_API_KEY'])
    print('\nМодели на сервере (readings):')
    for m in c.get_reading_models():
        print('  ', {k: m.get(k) for k in ('provider','model_name','reasoning_effort','uses_byok')})
except Exception as e:
    print('список с сервера не получен:', type(e).__name__, str(e)[:200])

## 6. Запуск двух судей (сетевые записи)
Две коллекции: «v0.4.4-full71-j1» и «v0.4.4-full71-j2». Скачаются два файла состояния, сохрани оба.

In [ ]:
!python protocol/docent_run.py launch --corpus full71 --judge "openai:gpt-5.6-terra:medium" --label v0.4.4-full71-j1
!python protocol/docent_run.py launch --corpus full71 --judge "openai:gpt-5.6-sol:medium" --label v0.4.4-full71-j2
from google.colab import files
files.download('docent_pilot_state_v0.4.4-full71-j1.json'); files.download('docent_pilot_state_v0.4.4-full71-j2.json')

## 7. Ожидание результатов
До 71 результата у каждого судьи; проверка раз в минуту до трёх часов. Можно перезапускать.

In [ ]:
import json, time
c = Docent(api_key=os.environ['DOCENT_API_KEY'])
states = {j: json.load(open(f'docent_pilot_state_v0.4.4-full71-{j}.json')) for j in ('j1','j2')}
for i in range(180):
    done = True
    for j, st in states.items():
        try:
            s = c.get_rubric_run_state(st['collection_id'], st['rubric_id'], include_failures=True)
            n = len(s.get('results') or [])
            print(time.strftime('%H:%M:%S'), j, 'результатов:', n, 'из 71')
            done = done and n >= 71
        except Exception as e:
            print(time.strftime('%H:%M:%S'), j, 'сервер не ответил:', type(e).__name__); done = False
    if done:
        print('готово'); break
    time.sleep(60)

## 8. Сырая выгрузка обоих судей
Если у кого-то меньше 71 и число больше не растёт полчаса, выгружай как есть: недостающие трассы пойдут в повтор (ячейка 10).

In [ ]:
!python protocol/docent_run.py raw-export --corpus full71 --label v0.4.4-full71-j1
!python protocol/docent_run.py raw-export --corpus full71 --label v0.4.4-full71-j2
files.download('raw_run_state_v0.4.4-full71-j1.json'); files.download('raw_run_state_v0.4.4-full71-j2.json')

## Если Colab сбросился
Заново ячейки 1–4, загрузи оба файла состояния в `/content/norm_ledger_v044/` (через `files.upload()`) и переходи к ячейкам 7–9. Ячейку 6 повторно не запускай.

## 9. Карта соответствия (без сетевых записей)
Сохраняет, какому исходному прогону соответствует каждый внутренний ID Docent. Пришли оба файла.

In [ ]:
import json
c = Docent(api_key=os.environ['DOCENT_API_KEY'])
for j in ('j1', 'j2'):
    st = json.load(open(f'docent_pilot_state_v0.4.4-full71-{j}.json'))
    raw = json.load(open(f'raw_run_state_v0.4.4-full71-{j}.json'))
    mp = {}
    for r in raw['results']:
        md = c.get_agent_run_metadata(st['collection_id'], r['agent_run_id'])
        mp[r['agent_run_id']] = md.get('source_run_id') or md.get('metadata', {}).get('source_run_id')
    json.dump(mp, open(f'id_map_v0.4.4-full71-{j}.json', 'w'), indent=1)
    print(j, len(mp), 'сопоставлено; без source_run_id:', [k for k, v in mp.items() if not v])
    files.download(f'id_map_v0.4.4-full71-{j}.json')

## 10. Один повтор для трасс без результата (по поправке v0.4.4)
Запускай только если в ячейке 7 у какого-то судьи осталось меньше 71 и число не растёт. Docent дооценивает недостающие трассы в той же коллекции; потом снова ячейки 7–9. Повтор – один раз.

In [ ]:
import json
c = Docent(api_key=os.environ['DOCENT_API_KEY'])
for j in ('j1', 'j2'):
    st = json.load(open(f'docent_pilot_state_v0.4.4-full71-{j}.json'))
    n = len(c.get_rubric_run_state(st['collection_id'], st['rubric_id'], include_failures=True).get('results') or [])
    if n < 71:
        jid = c.start_rubric_eval_job(st['collection_id'], st['rubric_id'], max_agent_runs=71, n_rollouts_per_input=1, include_metadata=False)
        print(j, 'было', n, '– повтор запущен, job', jid)
    else:
        print(j, 'полный, повтор не нужен')